# 02 Task Builder Contract Quotes

Este notebook audita cómo se construyó `tasks_quotes_prod.csv` en el run actual y define el contrato que debería cumplir un `Task Builder` limpio en la v2.

In [1]:
from __future__ import annotations

from pathlib import Path
import json
import pandas as pd

PROJECT_ROOT = Path(r"C:\TSIS_Data\v1\backtest_SmallCaps")
RUN_ID = "20260313_quotes_prod_full_12133_clean"
RUN_DIR = PROJECT_ROOT / "runs" / "polygon_realtime_audit" / RUN_ID
INPUTS_DIR = RUN_DIR / "inputs"

UNIVERSE_REFINED = PROJECT_ROOT / "data" / "reference" / "universe_pti" / "tickers_2005_2026_upper_excluding_ohlcv_1m_missing_vs_daily_ge_2B_or_null.parquet"
OFFICIAL_LIFECYCLE = PROJECT_ROOT / "data" / "reference" / "official_lifecycle_compiled.csv"
TASKS_CSV = INPUTS_DIR / "tasks_quotes_prod.csv"
TASKS_META_JSON = INPUTS_DIR / "tasks_quotes_prod_meta.json"
TICKERS_CSV = INPUTS_DIR / "tickers_quotes_prod.csv"

u = pd.read_parquet(UNIVERSE_REFINED) if UNIVERSE_REFINED.exists() else pd.DataFrame()
lc = pd.read_csv(OFFICIAL_LIFECYCLE) if OFFICIAL_LIFECYCLE.exists() else pd.DataFrame()
tasks = pd.read_csv(TASKS_CSV) if TASKS_CSV.exists() else pd.DataFrame()
tickers = pd.read_csv(TICKERS_CSV) if TICKERS_CSV.exists() else pd.DataFrame()
meta = json.loads(TASKS_META_JSON.read_text(encoding="utf-8")) if TASKS_META_JSON.exists() else {}

contract_snapshot = {
    "universe_refined_rows": int(len(u)),
    "universe_refined_unique_tickers": int(u["ticker"].astype(str).str.upper().nunique()) if "ticker" in u.columns else None,
    "official_lifecycle_rows": int(len(lc)),
    "tickers_prod_rows": int(len(tickers)),
    "tickers_prod_unique": int(tickers["ticker"].astype(str).str.upper().nunique()) if "ticker" in tickers.columns else None,
    "tasks_prod_rows": int(len(tasks)),
    "tasks_prod_unique_tickers": int(tasks["ticker"].astype(str).str.upper().nunique()) if "ticker" in tasks.columns else None,
    "tasks_meta": meta,
}
print(json.dumps(contract_snapshot, indent=2, ensure_ascii=False))

{
  "universe_refined_rows": 12133,
  "universe_refined_unique_tickers": 12133,
  "official_lifecycle_rows": 1970,
  "tickers_prod_rows": 1961,
  "tickers_prod_unique": 1961,
  "tasks_prod_rows": 3067682,
  "tasks_prod_unique_tickers": 1961,
  "tasks_meta": {
    "run_id": "20260313_quotes_prod_full_12133_clean",
    "universe_path": "C:\\TSIS_Data\\v1\\backtest_SmallCaps\\data\\reference\\universe_pti\\tickers_2005_2026_upper_excluding_ohlcv_1m_missing_vs_daily_ge_2B_or_null.parquet",
    "lifecycle_path": "C:\\TSIS_Data\\v1\\backtest_SmallCaps\\data\\reference\\official_lifecycle_compiled.csv",
    "tickers_count": 1961,
    "tasks_total": 3067682,
    "date_min": "1995-09-13",
    "date_max": "2026-03-13"
  }
}


## Snapshot Del Run Real

Esta sección describe lo que ocurrió realmente en el run auditado.

In [2]:
if len(tasks):
    d = pd.to_datetime(tasks["date"], errors="coerce")
    audit = {
        "date_min": str(d.min().date()) if d.notna().any() else None,
        "date_max": str(d.max().date()) if d.notna().any() else None,
        "pre_2005_rows": int((d < pd.Timestamp("2005-01-01")).sum()),
        "post_2025_rows": int((d > pd.Timestamp("2025-12-31")).sum()),
        "duplicates_task_key": int(tasks.duplicated(["ticker", "date"]).sum()),
    }
    print(json.dumps(audit, indent=2))
    display(tasks.loc[d < pd.Timestamp("2005-01-01")].head(10))
    display(tasks.loc[d > pd.Timestamp("2025-12-31")].head(10))
else:
    print("tasks_quotes_prod.csv no existe o está vacío")

{
  "date_min": "1995-09-13",
  "date_max": "2026-03-13",
  "pre_2005_rows": 59354,
  "post_2025_rows": 74019,
  "duplicates_task_key": 0
}


,ticker,date
77710,AIMD,1996-05-30
77711,AIMD,1996-05-31
77712,AIMD,1996-06-03
77713,AIMD,1996-06-04
77714,AIMD,1996-06-05
77715,AIMD,1996-06-06
77716,AIMD,1996-06-07
77717,AIMD,1996-06-10
77718,AIMD,1996-06-11
77719,AIMD,1996-06-12


,ticker,date
599,AACI,2026-02-17
600,AACI,2026-02-18
601,AACI,2026-02-19
602,AACI,2026-02-20
603,AACI,2026-02-23
604,AACI,2026-02-24
605,AACI,2026-02-25
606,AACI,2026-02-26
607,AACI,2026-02-27
608,AACI,2026-03-02


## Contrato Esperado De V2

Esta sección define cómo debería comportarse un `Task Builder` limpio en la v2.

In [3]:
task_builder_contract = pd.DataFrame([
    {"field": "input_universe", "expected": "universo refinado versionado e inmutable"},
    {"field": "input_lifecycle", "expected": "official_lifecycle_compiled.csv congelado para el run"},
    {"field": "date_policy", "expected": "calendar oficial XNYS, no bdate_range"},
    {"field": "date_window", "expected": "date_from/date_to explícitos del run"},
    {"field": "task_key", "expected": "ticker|date"},
    {"field": "outputs", "expected": "tasks_quotes_prod.csv + tasks_quotes_prod.meta.json + tickers_quotes_prod.csv"},
    {"field": "forbidden", "expected": "no construir inputs productivos desde celdas manuales"},
])
task_builder_contract

,field,expected
0,input_universe,universo refinado versionado e inmutable
1,input_lifecycle,official_lifecycle_compiled.csv congelado para...
2,date_policy,"calendar oficial XNYS, no bdate_range"
3,date_window,date_from/date_to explícitos del run
4,task_key,ticker|date
5,outputs,tasks_quotes_prod.csv + tasks_quotes_prod.meta...
6,forbidden,no construir inputs productivos desde celdas m...


In [4]:
v2_rules = pd.DataFrame([
    {
        "confirmed_problem": "tasks construidas con bdate_range en vez de calendario oficial",
        "expected_v2_rule": "usar calendario oficial XNYS y rango explícito del run",
    },
    {
        "confirmed_problem": "tasks fuera de 2005-2025 por no recortar ventana",
        "expected_v2_rule": "aplicar date_from/date_to antes de expandir ticker,date",
    },
    {
        "confirmed_problem": "materialización productiva desde celda manual en notebook operativo",
        "expected_v2_rule": "script dedicado build_quotes_run_inputs.py con outputs versionados",
    },
])
v2_rules

,confirmed_problem,expected_v2_rule
0,tasks construidas con bdate_range en vez de ca...,usar calendario oficial XNYS y rango explícito...
1,tasks fuera de 2005-2025 por no recortar ventana,aplicar date_from/date_to antes de expandir ti...
2,materialización productiva desde celda manual ...,script dedicado build_quotes_run_inputs.py con...
